# Ordered Logistic Regression Results: FAIRˆ² Dataset Exploration with `mlcroissant`
This notebook demonstrates how to explore, process, and visualize the FAIRˆ² dataset using the `mlcroissant` library. All dataset elements are referenced by their `@id` fields, following the Croissant schema.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load overall dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and show summary information
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {getattr(metadata, 'name', '<no name>')}")
print(f"Description: {getattr(metadata, 'description', '<no description>')}")
print(f"Version: {getattr(metadata, 'version', '<no version>')}")
print(f"Identifier: {getattr(metadata, 'identifier', '<no identifier>')}")

## 2. Data Overview
Review and list available record sets, their IDs, and the fields available within each record set. All references use the `@id` field.

In [ ]:
# List record sets (by @id), and for each, list their fields and columns (by @id)
record_sets = list(dataset.record_sets())  # Each is a RecordSet object
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    record_set_ids.append(rs_id)
    name = getattr(rs, 'name', '(no name)')
    print(f"Record set @id: {rs_id}")
    print(f"  Name: {name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field @id: {getattr(field, '@id', None)}")
            print(f"      Name: {getattr(field, 'name', '')}")
            # Show columns for each field if present
            if hasattr(field, 'columns') and field.columns:
                print("      Columns:")
                for col in field.columns:
                    print(f"        - Column @id: {getattr(col, '@id', None)} | Name: {getattr(col, 'name', '')}")
    print("")
# Save IDs for subsequent cells
record_set_ids

## 3. Data Extraction
Load data from a selected record set into a DataFrame for analysis. Use the `@id` of the record set from the previous section.

In [ ]:
dataframes = {}

# Example: extract all record sets into pandas DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# For demonstration, pick the first available record set with records
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f"Using record set: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps such as filtering, normalizing numeric fields, and grouping. All field references use their `@id` values for clarity and reproducibility.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Identify a numeric field by inspecting dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use the first numeric field
        print(f"Analyzing numeric field (by @id): {numeric_field_id}")

        threshold = df[numeric_field_id].mean()  # Use mean as threshold for demo
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (threshold: mean value)")
        print(filtered_df.head())

        # Normalize the field
        mean_val = filtered_df[numeric_field_id].mean()
        std_val = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by a categorical field (by @id)
        # Find a likely candidate for grouping: non-numeric field
        group_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No main record set data available for EDA.")

## 5. Visualization
Visualize data distributions for numeric and categorical fields, referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Plot distribution of a numeric field
    if 'numeric_field_id' in locals() and numeric_field_id in df:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of field (@id): {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

    # Categorical breakdown (bar chart) for group_field_id if available
    if 'group_field_id' in locals() and group_field_id in df:
        plt.figure(figsize=(10,4))
        value_counts = df[group_field_id].value_counts().head(15)
        sns.barplot(x=value_counts.index, y=value_counts.values)
        plt.title(f"Top {len(value_counts)} values in field (@id): {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel("Count")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
We've demonstrated how to load, explore, and analyze a FAIRˆ² Croissant dataset using the `mlcroissant` library, referencing all entities by their `@id`. Follow similar steps to inspect specific fields and perform more advanced analyses customized to your use case.